# Learning Word Embeddings

## 1. Core Idea: Learn Word Representations from Context

After understanding what a word embedding is, the next question is:

> **How do we actually learn the embedding vectors from text?**

The key idea is that we do not manually assign semantic information to each word.

Instead, we construct a **prediction objective** from a large text corpus and allow gradient descent to learn embeddings that are useful for solving that objective.

The general process is:

$$
\boxed{
\text{Text Corpus}
\rightarrow
\text{Context--Target Relationships}
\rightarrow
\text{Prediction Task}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradient}
\rightarrow
\text{Embedding Update}
}
$$

Suppose the corpus contains:

> I drink coffee every morning.

We can construct a training example such as:

$$
\text{Context}
\rightarrow
\text{Target}
$$

For example:

$$
\text{"I drink"}
\rightarrow
\text{"coffee"}
$$

or, depending on the embedding-learning method:

$$
\text{"coffee"}
\rightarrow
\text{"drink"}
$$

The exact definition of context and target depends on the training objective.

The fundamental principle is:

$$
\boxed{
\text{Words that participate in similar linguistic contexts}
\rightarrow
\text{similar learned representations}
}
$$

Therefore, the embedding is learned **indirectly through prediction** rather than being explicitly given semantic labels.

---

## 2. From a Text Corpus to Training Examples

Suppose we have:

$$
[\text{The},\text{cat},\text{drinks},\text{milk}]
$$

We can construct word relationships from this sequence.

For example:

$$
(\text{cat},\text{drinks})
$$

may be treated as a positive word-context relationship.

A corpus therefore becomes a large collection of training pairs:

$$
(x,y)
$$

where $x$ represents some context information and $y$ represents a target word or word-context relationship.

The target word can be represented as a one-hot vector:

$$
y\in\mathbb R^V
$$

while the input word is represented using an embedding:

$$
e_w\in\mathbb R^d
$$

with:

$$
e_w=Eo_w
$$

where:

$$
E\in\mathbb R^{d\times V}
$$

This gives a very important insight:

> **Unlabeled text can be turned into training supervision by exploiting the relationships that already exist inside the text.**

We do not need a human annotator to provide a semantic label such as:

$$
\text{cat}\rightarrow\text{animal}
$$

Instead, the text itself provides the learning signal through context and co-occurrence.

---

## 3. Word2Vec and the Skip-gram Objective

One of the most important approaches is **Word2Vec**.

For the Course 5 perspective, the key idea is **Skip-gram**.

The Skip-gram objective is:

$$
\boxed{
\text{Center Word}
\rightarrow
\text{Predict Context Word}
}
$$

Suppose we have:

> The cat drinks milk.

Choose:

$$
\text{center word}=\text{cat}
$$

and consider the words around it.

Possible training pairs include:

$$
(\text{cat},\text{The})
$$

and:

$$
(\text{cat},\text{drinks})
$$

The model therefore learns to predict words that occur around the center word.

Formally, if:

- $w_c$ = center word;
- $w_o$ = context word;

then the objective is to model:

$$
\boxed{
P(w_o\mid w_c)
}
$$

The important mechanism is:

$$
w_c
\rightarrow
e_c
\rightarrow
\text{prediction of }w_o
$$

Now consider two center words, such as:

$$
\text{cat}
\quad\text{and}\quad
\text{dog}
$$

If both frequently appear in similar contexts, they are repeatedly required to perform similar prediction behavior.

Therefore, training can encourage:

$$
e_{\text{cat}}
\approx
e_{\text{dog}}
$$

This gives the central causal chain:

$$
\boxed{
\text{Similar Contexts}
\rightarrow
\text{Similar Prediction Behavior}
\rightarrow
\text{Similar Embeddings}
}
$$

This is how semantic structure can emerge without explicitly teaching semantic labels.

---

## 4. Why a Direct Softmax Is Expensive

A straightforward implementation would predict over the entire vocabulary.

Suppose:

$$
e_c\in\mathbb R^d
$$

is the embedding of the center word.

We could transform it into scores for all $V$ words:

$$
z\in\mathbb R^V
$$

and then apply softmax:

$$
\hat y_i
=
\frac{e^{z_i}}
{\sum_{j=1}^{V}e^{z_j}}
$$

The output is:

$$
\hat y\in\mathbb R^V
$$

and the loss could be categorical cross-entropy:

$$
L
=
-\sum_{i=1}^{V}
y_i\log\hat y_i
$$

The problem is the vocabulary size.

If:

$$
V=100,000
$$

then every prediction requires computations involving approximately $100,000$ possible words.

For very large vocabularies, this becomes expensive.

Therefore, Word2Vec can use:

$$
\boxed{
\text{Negative Sampling}
}
$$

to make the learning problem much cheaper.

---

## 5. Negative Sampling

Negative sampling changes the prediction problem.

Instead of asking:

> **Which one of all $V$ words is the correct context word?**

we ask:

> **Is this particular word pair a real context relationship or not?**

For a positive pair:

$$
(\text{cat},\text{drinks})
$$

we define:

$$
y=1
$$

Then we randomly sample several words that should act as negative examples.

For example:

$$
(\text{cat},\text{computer})
$$

$$
(\text{cat},\text{car})
$$

$$
(\text{cat},\text{mountain})
$$

with:

$$
y=0
$$

Therefore:

$$
\boxed{
\text{Positive Pair}
\rightarrow
y=1
}
$$

and:

$$
\boxed{
\text{Negative Pair}
\rightarrow
y=0
}
$$

Instead of a $V$-class classification problem, we now have several small binary classification problems.

---

## 6. The Negative-Sampling Prediction Function

Let:

$$
e_c\in\mathbb R^d
$$

be the center-word embedding and:

$$
e_o\in\mathbb R^d
$$

be the candidate context-word embedding.

First compute their dot product:

$$
s=e_c^\top e_o
$$

Then apply the sigmoid:

$$
\boxed{
\hat y
=
\sigma(e_c^\top e_o)
}
$$

where:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Interpretation:

- large positive $e_c^\top e_o$ → $\hat y$ approaches $1$;
- large negative $e_c^\top e_o$ → $\hat y$ approaches $0$.

The binary cross-entropy loss for one pair is:

$$
\boxed{
L
=
-
\left[
y\log\hat y
+
(1-y)\log(1-\hat y)
\right]
}
$$

The model therefore learns to distinguish:

$$
\boxed{
\text{Real Context Relationship}
}
$$

from:

$$
\boxed{
\text{Random / Negative Relationship}
}
$$

---

## 7. How Negative Sampling Shapes the Embedding Space

This is the most important mathematical intuition in the topic.

Suppose we have a positive pair:

$$
(c,o)
$$

with:

$$
y=1
$$

If:

$$
e_c^\top e_o
$$

is too small, then:

$$
\hat y\approx0
$$

and the loss is large.

Gradient descent therefore creates pressure to increase:

$$
e_c^\top e_o
$$

which tends to make the two vectors more aligned.

Conceptually:

$$
\boxed{
\text{Positive Pair}
\rightarrow
\text{Increase Compatibility}
}
$$

Now consider a negative pair:

$$
(c,n)
$$

with:

$$
y=0
$$

If:

$$
e_c^\top e_n
$$

is too large, then:

$$
\hat y\approx1
$$

which is incorrect.

The optimization therefore creates pressure to reduce their compatibility.

Conceptually:

$$
\boxed{
\text{Negative Pair}
\rightarrow
\text{Decrease Compatibility}
}
$$

Thus, over many training examples:

$$
\boxed{
\text{Positive Relationships}
\rightarrow
\text{Pull Compatible Representations Together}
}
$$

while:

$$
\boxed{
\text{Negative Relationships}
\rightarrow
\text{Push Incompatible Representations Apart}
}
$$

This repeated process shapes the geometry of the embedding space.

The overall mechanism becomes:

$$
\boxed{
\text{Context Statistics}
\rightarrow
\text{Positive/Negative Constraints}
\rightarrow
\text{Gradient Updates}
\rightarrow
\text{Embedding Geometry}
}
$$

This is the underlying reason word embeddings can develop semantic structure.

---

## 8. Connection Back to the Embedding Matrix

Recall that all word vectors are stored in:

$$
E\in\mathbb R^{d\times V}
$$

where:

$$
e_i=E_{:,i}
$$

For a center word and candidate context word, the model extracts:

$$
e_c
\quad\text{and}\quad
e_o
$$

and computes:

$$
e_c^\top e_o
$$

Then:

$$
e_c^\top e_o
\rightarrow
\sigma(\cdot)
\rightarrow
L
$$

During backpropagation:

$$
L
\rightarrow
\frac{\partial L}{\partial e_c},
\frac{\partial L}{\partial e_o}
\rightarrow
\frac{\partial L}{\partial E}
$$

and the embedding matrix is updated:

$$
\boxed{
E
\leftarrow
E-\alpha\frac{\partial L}{\partial E}
}
$$

Therefore, learning word embeddings is simply another optimization problem in which the embedding matrix is a set of trainable parameters.

The full loop is:

$$
\boxed{
\text{Lookup}
\rightarrow
\text{Prediction}
\rightarrow
\text{Loss}
\rightarrow
\text{Backward}
\rightarrow
\text{Update Embeddings}
}
$$

---

## 9. GloVe: Learning from Global Co-occurrence Statistics

Another important approach is **GloVe (Global Vectors for Word Representation)**.

The intuition is slightly different from the local prediction view of Skip-gram.

GloVe makes strong use of **global word co-occurrence statistics**.

Define a co-occurrence matrix:

$$
X_{ij}
$$

where $X_{ij}$ represents how often word $j$ appears in the context of word $i$.

The idea is:

$$
\boxed{
\text{Co-occurrence Statistics}
\rightarrow
\text{Word Representations}
}
$$

A simplified conceptual view is that the learned vectors should reflect relationships found in these corpus-wide statistics.

The GloVe objective can be written as:

$$
J=
\sum_{i,j}
f(X_{ij})
\left(
e_i^\top\tilde e_j
+b_i+\tilde b_j
-\log X_{ij}
\right)^2
$$

where:

- $e_i$: word vector of word $i$;
- $\tilde e_j$: context vector of word $j$;
- $b_i,\tilde b_j$: bias terms;
- $X_{ij}$: co-occurrence count;
- $f(X_{ij})$: weighting function.

The most important point is not memorizing the exact objective.

It is understanding the difference in perspective:

$$
\boxed{
\text{Word2Vec}
\rightarrow
\text{Learn from Prediction Relationships}
}
$$

while:

$$
\boxed{
\text{GloVe}
\rightarrow
\text{Learn from Global Co-occurrence Statistics}
}
$$

Both aim to learn useful continuous representations of words.

---

# Final Mental Model

The whole process of **Learning Word Embeddings** can be summarized as:

$$
\boxed{
\text{Large Text Corpus}
\rightarrow
\text{Word/Context Relationships}
\rightarrow
\text{Learning Objective}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradient Descent}
\rightarrow
\text{Embedding Matrix}
}
$$

For **Skip-gram**:

$$
\boxed{
\text{Center Word}
\rightarrow
\text{Predict Context Word}
}
$$

For **Negative Sampling**:

$$
\boxed{
\text{Positive Pair}
\rightarrow
\text{High Compatibility}
}
$$

$$
\boxed{
\text{Negative Pair}
\rightarrow
\text{Low Compatibility}
}
$$

with:

$$
\hat y
=
\sigma(e_c^\top e_o)
$$

For **GloVe**:

$$
\boxed{
\text{Global Co-occurrence Statistics}
\rightarrow
\text{Learned Word Vectors}
}
$$

The deepest idea to remember is:

$$
\boxed{
\text{We do not explicitly teach the model the meaning of each word.}
}
$$

Instead:

$$
\boxed{
\text{We design a learning objective based on language usage}
\rightarrow
\text{the model learns the representations}
}
$$

and eventually:

$$
\boxed{
\text{Statistical Structure of Language}
\rightarrow
\text{Geometry of Embedding Space}
}
$$

That is the fundamental mechanism behind **Learning Word Embeddings**.

# Word2Vec — Skip-gram and CBOW

Word2Vec is a method for learning **distributed word representations** from a large text corpus. Instead of directly assigning semantic information to each word, Word2Vec constructs a prediction task from the **relationship between a word and its surrounding context**, then learns the embedding matrix through optimization.

The core pipeline is:

$$
\boxed{
\text{Raw Text}
\rightarrow
\text{Context Window}
\rightarrow
\text{Training Pairs}
\rightarrow
\text{Prediction Objective}
\rightarrow
\text{Learned Word Embeddings}
}
$$

Word2Vec mainly contains two architectures:

$$
\boxed{
\text{CBOW}
\qquad\text{and}\qquad
\text{Skip-gram}
}
$$

Their fundamental difference is the **direction of prediction**:

$$
\boxed{
\text{Skip-gram: Center}\rightarrow\text{Context}
}
$$

$$
\boxed{
\text{CBOW: Context}\rightarrow\text{Center}
}
$$

---

## 1. Core Idea and Context Window

The underlying intuition is the **distributional hypothesis**:

> A word's meaning is related to the words that tend to appear around it.

For example, consider:

> I drink coffee every morning.

The word `coffee` frequently appears in contexts such as:

> drink coffee

> coffee every morning

> hot coffee

Instead of explicitly labeling:

$$
\text{coffee}\rightarrow\text{beverage}
$$

we use the surrounding words themselves to construct a learning problem.

Thus:

$$
\boxed{
\text{Context}
\rightarrow
\text{Prediction Signal}
\rightarrow
\text{Learned Representation}
}
$$

Suppose the sentence is:

> The quick brown fox jumps over the lazy dog.

We choose:

$$
\text{window size}=2
$$

and select `fox` as the center word.

The context window is:

$$
[\text{quick},\text{brown},\boxed{\text{fox}},\text{jumps},\text{over}]
$$

Therefore, the words surrounding `fox` are:

$$
\{\text{quick},\text{brown},\text{jumps},\text{over}\}
$$

This gives the relationship:

$$
\boxed{
\text{fox}
\leftrightarrow
\{\text{quick},\text{brown},\text{jumps},\text{over}\}
}
$$

From exactly this same context window, Skip-gram and CBOW construct **different training examples**.

That is the key distinction to understand.

---

# 2. Skip-gram

Skip-gram uses the center word to predict each of its context words:

$$
\boxed{
\text{Center Word}
\rightarrow
\text{Context Word}
}
$$

For the previous example:

> The quick brown **fox** jumps over the lazy dog.

with:

$$
\text{window size}=2
$$

and center word:

$$
w_c=\text{fox}
$$

the context words are:

$$
\text{quick},\text{brown},\text{jumps},\text{over}
$$

Skip-gram creates **one training pair for each context word**:

$$
(\text{fox},\text{quick})
$$

$$
(\text{fox},\text{brown})
$$

$$
(\text{fox},\text{jumps})
$$

$$
(\text{fox},\text{over})
$$

So one center word generates multiple training examples:

$$
\boxed{
\text{One Center Word}
\rightarrow
\text{Multiple Context Targets}
}
$$

More generally, if the center word is $w_c$ and its context words are $w_{o_1},\ldots,w_{o_m}$, then Skip-gram generates:

$$
\boxed{
(w_c,w_{o_1}),
(w_c,w_{o_2}),
\ldots,
(w_c,w_{o_m})
}
$$

### Skip-gram Architecture

![Skip-gram Architecture](https://nguyentruonglong.net/images/SkipGram.png)

### A complete small example

Consider:

> I love machine learning.

and:

$$
\text{window size}=1
$$

For center word `love`, the neighboring words are:

$$
[\text{I},\boxed{\text{love}},\text{machine}]
$$

so Skip-gram produces:

$$
(\text{love},\text{I})
$$

$$
(\text{love},\text{machine})
$$

For center word `machine`:

$$
[\text{love},\boxed{\text{machine}},\text{learning}]
$$

so:

$$
(\text{machine},\text{love})
$$

$$
(\text{machine},\text{learning})
$$

For `learning`, assuming the boundary leaves only `machine` in the window:

$$
(\text{learning},\text{machine})
$$

Therefore, the corpus is converted into many center-context prediction examples.

### Mathematical objective

Let:

$$
w_c=\text{center word}
$$

and:

$$
w_o=\text{context word}
$$

The model tries to learn:

$$
\boxed{
P(w_o\mid w_c)
}
$$

Suppose the center word $w_c$ is represented by an input embedding:

$$
v_{w_c}\in\mathbb R^d
$$

and every candidate context word $w$ has an output embedding:

$$
v_w\in\mathbb R^d
$$

The compatibility between the center word and a candidate context word is measured by their dot product:

$$
s(w,w_c)=v_w^Tv_{w_c}
$$

A larger value of $v_w^Tv_{w_c}$ means that the model considers $w$ more compatible with the center word $w_c$.

However, the dot product itself is not a probability.

To convert the scores of all vocabulary words into a probability distribution, Word2Vec can use the **softmax function**:

$$
\boxed{
P(w_o\mid w_c)
=
\frac{\exp(v_{w_o}^Tv_{w_c})}
{\sum_{w=1}^{V}\exp(v_w^Tv_{w_c})}
}
$$

This formula should be understood carefully.

The **numerator** is:

$$
\exp(v_{w_o}^Tv_{w_c})
$$

which is the exponentiated score of the actual context word $w_o$.

The **denominator** is:

$$
\sum_{w=1}^{V}\exp(v_w^Tv_{w_c})
$$

which sums the exponentiated scores of **all $V$ words in the vocabulary**.

Therefore, the probability of the correct context word is its score divided by the total score of every possible vocabulary word.

In other words:

$$
\boxed{
\text{Probability of correct context}
=
\frac{\text{score of correct word}}
{\text{scores of all possible words}}
}
$$

This guarantees that:

$$
\sum_{w=1}^{V}P(w\mid w_c)=1
$$

so the model produces a valid probability distribution over the vocabulary.

At the high level:

$$
\boxed{
w_c
\rightarrow
v_{w_c}
\rightarrow
\text{Scores for all vocabulary words}
\rightarrow
\text{Softmax}
\rightarrow
P(w_o\mid w_c)
}
$$

### Training objective

The goal of Skip-gram is to assign high probability to the context words that actually occur around the center word.

For a sequence of center words, the log-likelihood can be written conceptually as:

$$
J
=
\sum_c
\sum_{\substack{-m\le j\le m\\j\neq0}}
\log P(w_{c+j}\mid w_c)
$$

Substituting the softmax probability:

$$
J
=
\sum_c
\sum_{\substack{-m\le j\le m\\j\neq0}}
\log
\left(
\frac{
\exp(v_{w_{c+j}}^Tv_{w_c})
}{
\sum_{w=1}^{V}
\exp(v_w^Tv_{w_c})
}
\right)
$$

Training maximizes $J$, or equivalently minimizes:

$$
L=-J
$$

For a single center-context pair $(w_c,w_o)$, the loss is:

$$
\boxed{
L
=
-\log P(w_o\mid w_c)
}
$$

Using the softmax formula:

$$
\boxed{
L
=
-\log
\left(
\frac{
\exp(v_{w_o}^Tv_{w_c})
}{
\sum_{w=1}^{V}
\exp(v_w^Tv_{w_c})
}
\right)
}
$$

The effect of this loss is intuitive.

For the correct context word $w_o$, we want:

$$
v_{w_o}^Tv_{w_c}
$$

to become relatively large.

At the same time, the model must compare this score against the scores of the other vocabulary words.

Thus the embedding is learned through the prediction task:

$$
\boxed{
\text{Center Word}
\rightarrow
\text{Predict Context}
\rightarrow
\text{Prediction Error}
\rightarrow
\text{Gradient}
\rightarrow
\text{Embedding Update}
}
$$

The important idea is:

> The embedding of a word is learned because it must be useful for predicting the words that occur around that word.

### Computational limitation

The main limitation of this formulation is the denominator:

$$
\sum_{w=1}^{V}\exp(v_w^Tv_{w_c})
$$

For every training example, the model must evaluate the score for **every word in the vocabulary**.

If the vocabulary size is $V$ and the embedding dimension is $d$, the computation is approximately:

$$
O(Vd)
$$

per prediction.

When $V$ is very large, this becomes computationally expensive.

For example, if:

$$
V=100000
$$

then a single prediction requires considering up to $100000$ candidate words.

Since Word2Vec is trained on a very large number of training examples, this repeated computation can make training slow.

Therefore, the limitation can be summarized as:

$$
\boxed{
\text{Large Vocabulary}
\rightarrow
\text{Many Score Computations}
\rightarrow
\text{High Computational Cost}
\rightarrow
\text{Slow Training}
}
$$

---

# 3. CBOW

CBOW stands for **Continuous Bag-of-Words**.

It reverses the prediction direction:

$$
\boxed{
\text{Context Words}
\rightarrow
\text{Center Word}
}
$$

Using the same sentence:

> The quick brown **fox** jumps over the lazy dog.

with window size $2$, the context of `fox` is:

$$
\{\text{quick},\text{brown},\text{jumps},\text{over}\}
$$

CBOW combines these context words and uses them to predict:

$$
\text{fox}
$$

Thus the training example is:

$$
\boxed{
[
\text{quick},
\text{brown},
\text{jumps},
\text{over}
]
\rightarrow
\text{fox}
}
$$

This is different from Skip-gram in an important way.

Skip-gram:

$$
\text{fox}
\rightarrow
\begin{cases}
\text{quick}\\
\text{brown}\\
\text{jumps}\\
\text{over}
\end{cases}
$$

CBOW:

$$
\{\text{quick},\text{brown},\text{jumps},\text{over}\}
\rightarrow
\text{fox}
$$

So:

$$
\boxed{
\text{Skip-gram: One Center}\rightarrow\text{Many Context Predictions}
}
$$

while:

$$
\boxed{
\text{CBOW: Many Contexts}\rightarrow\text{One Center Prediction}
}
$$

### CBOW Architecture

![CBOW Architecture](https://nguyentruonglong.net/images/GeneralCBOW.png)

### A complete small example

Again consider:

> I love machine learning.

with:

$$
\text{window size}=1
$$

For center word `love`:

$$
[\text{I},\boxed{\text{love}},\text{machine}]
$$

CBOW creates:

$$
\boxed{
[\text{I},\text{machine}]
\rightarrow
\text{love}
}
$$

For center word `machine`:

$$
[\text{love},\boxed{\text{machine}},\text{learning}]
$$

we get:

$$
\boxed{
[\text{love},\text{learning}]
\rightarrow
\text{machine}
}
$$

For `learning`:

$$
[\text{machine}]
\rightarrow
\text{learning}
$$

Thus each center position produces **one prediction example** containing multiple context words.

### How CBOW combines the context words

Suppose the context contains $m$ words:

$$
w_1,w_2,\ldots,w_m
$$

and their embeddings are:

$$
e_1,e_2,\ldots,e_m
$$

A simple CBOW representation is the average:

$$
\boxed{
\bar e
=
\frac{1}{m}
\sum_{i=1}^{m}e_i
}
$$

Then:

$$
\bar e
\rightarrow
\text{Prediction of Center Word}
$$

The word `"bag"` in CBOW refers to the fact that the context words are treated collectively rather than as an ordered sequence in the input representation.

The important point is not that the model literally throws the context words into a random unordered collection, but that their embeddings are aggregated into a representation used to predict the center word.

### Mathematical objective

The CBOW model tries to learn:

$$
\boxed{
P
\left(
w_c
\mid
w_{c-m},\ldots,w_{c-1},w_{c+1},\ldots,w_{c+m}
\right)
}
$$

After aggregating the context embeddings into $\bar e$, the model can score each possible vocabulary word $w$ using:

$$
v_w^T\bar e
$$

The softmax probability of the correct center word $w_c$ is:

$$
\boxed{
P(w_c\mid\text{context})
=
\frac{
\exp(v_{w_c}^T\bar e)
}{
\sum_{w=1}^{V}
\exp(v_w^T\bar e)
}
}
$$

Again, the numerator represents the score of the correct center word:

$$
\exp(v_{w_c}^T\bar e)
$$

while the denominator sums the scores of **all $V$ vocabulary words**:

$$
\sum_{w=1}^{V}\exp(v_w^T\bar e)
$$

The loss for one training example is:

$$
\boxed{
L
=
-\log P(w_c\mid\text{context})
}
$$

or explicitly:

$$
\boxed{
L
=
-\log
\left(
\frac{
\exp(v_{w_c}^T\bar e)
}{
\sum_{w=1}^{V}
\exp(v_w^T\bar e)
}
\right)
}
$$

Thus the learning process is:

$$
\boxed{
\text{Context Words}
\rightarrow
\text{Embeddings}
\rightarrow
\text{Aggregate}
\rightarrow
\bar e
\rightarrow
\text{Predict Center}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradient Update}
}
$$

### Computational limitation

Although CBOW uses several context words together and produces only one target prediction per window, the final prediction still uses a softmax over the entire vocabulary.

The denominator is:

$$
\sum_{w=1}^{V}
\exp(v_w^T\bar e)
$$

Therefore, for every CBOW training example, the model still has to compute scores for all $V$ vocabulary words.

Its computational cost is approximately:

$$
O(Vd)
$$

per prediction.

Hence a large vocabulary also makes CBOW computationally expensive:

$$
\boxed{
\text{Large Vocabulary}
\rightarrow
\text{Full Vocabulary Softmax}
\rightarrow
\text{Many Computations}
\rightarrow
\text{Slow Training}
}
$$

The difference between Skip-gram and CBOW is therefore mainly in **how training examples are constructed and what is predicted**. The large-vocabulary softmax remains a computational challenge in both cases.

---

# 4. Skip-gram vs. CBOW

Take the sentence:

> The cat drinks milk.

and choose:

$$
\text{window size}=1
$$

Take `drinks` as the center.

The local window is:

$$
[\text{cat},\boxed{\text{drinks}},\text{milk}]
$$

### Skip-gram

Create two pairs:

$$
(\text{drinks},\text{cat})
$$

$$
(\text{drinks},\text{milk})
$$

So:

$$
\boxed{
\text{drinks}
\rightarrow
\text{cat}
}
$$

$$
\boxed{
\text{drinks}
\rightarrow
\text{milk}
}
$$

### CBOW

Create one example:

$$
\boxed{
[\text{cat},\text{milk}]
\rightarrow
\text{drinks}
}
$$

Therefore, the fundamental difference is not merely the arrows.

It is the **structure of the training examples**:

$$
\boxed{
\text{Skip-gram}
=
\text{One center + multiple separate targets}
}
$$

while:

$$
\boxed{
\text{CBOW}
=
\text{Multiple context inputs + one target}
}
$$

### Direct Comparison

| Property | Skip-gram | CBOW |
|---|---|---|
| Input | Center word | Context words |
| Target | Context word(s) | Center word |
| Direction | Center → Context | Context → Center |
| Training examples | One center generates multiple pairs | One window generates one example |
| Main idea | Predict surrounding words | Predict the missing center word |
| Context handling | Each context word is a separate target | Context words are combined |
| Core probability | $P(w_o\mid w_c)$ | $P(w_c\mid\text{context})$ |
| Softmax | Over all $V$ context candidates | Over all $V$ center-word candidates |

The easiest way to remember them is:

$$
\boxed{
\text{Skip-gram}:
\text{What words tend to appear around this word?}
}
$$

$$
\boxed{
\text{CBOW}:
\text{What word fits this surrounding context?}
}
$$

---

# 5. How Word2Vec Learns the Embedding Matrix

Recall that all word embeddings are stored in:

$$
E\in\mathbb R^{d\times V}
$$

For Skip-gram, a center word $w_c$ is mapped to:

$$
e_c=E_{:,c}
$$

and used to predict context words.

For CBOW, the context words are mapped to:

$$
e_1,e_2,\ldots,e_m
$$

and aggregated:

$$
\bar e
=
\frac{1}{m}\sum_{i=1}^{m}e_i
$$

and used to predict the center word.

The prediction generates a loss:

$$
L
$$

and backpropagation computes gradients with respect to the embeddings.

Therefore:

$$
\boxed{
\text{Context Relationships}
\rightarrow
\text{Prediction Error}
\rightarrow
\text{Gradients}
\rightarrow
\text{Embedding Updates}
}
$$

and:

$$
\boxed{
E
\leftarrow
E-\alpha\frac{\partial L}{\partial E}
}
$$

During training, the embeddings are continuously adjusted so that words occurring in useful contextual relationships become represented in compatible directions in the embedding space.

After a very large number of training examples, the embedding matrix becomes a representation of statistical regularities in the language.

This is why semantic structure can emerge:

$$
\boxed{
\text{Similar Context Patterns}
\rightarrow
\text{Similar Learning Signals}
\rightarrow
\text{Similar Embeddings}
}
$$

The Word2Vec model therefore does not directly learn a dictionary definition for every word.

It learns representations that are useful for **predicting linguistic context**.

---

# Final Mental Model

Word2Vec starts from raw text and turns it into a prediction problem:

$$
\boxed{
\text{Raw Text}
\rightarrow
\text{Sliding Context Window}
\rightarrow
\text{Training Examples}
}
$$

Then there are two possibilities:

$$
\boxed{
\text{Skip-gram}
:
\text{Center}
\rightarrow
\text{Context}
}
$$

or:

$$
\boxed{
\text{CBOW}
:
\text{Context}
\rightarrow
\text{Center}
}
$$

For a center word with multiple surrounding words:

$$
\boxed{
\text{Skip-gram}
=
\text{One Center}
\rightarrow
\text{Several Training Pairs}
}
$$

whereas:

$$
\boxed{
\text{CBOW}
=
\text{Several Context Words}
\rightarrow
\text{One Training Example}
}
$$

For Skip-gram, the prediction probability is:

$$
\boxed{
P(w_o\mid w_c)
=
\frac{
\exp(v_{w_o}^Tv_{w_c})
}{
\sum_{w=1}^{V}\exp(v_w^Tv_{w_c})
}
}
$$

For CBOW, after combining the context embeddings into $\bar e$:

$$
\boxed{
P(w_c\mid\text{context})
=
\frac{
\exp(v_{w_c}^T\bar e)
}{
\sum_{w=1}^{V}\exp(v_w^T\bar e)
}
}
$$

In both cases, the denominator sums over the entire vocabulary:

$$
\boxed{
\sum_{w=1}^{V}
}
$$

Therefore, the central computational limitation is:

$$
\boxed{
V\text{ becomes very large}
\rightarrow
\text{many score computations per training example}
\rightarrow
\text{high computational cost}
\rightarrow
\text{slow training}
}
$$

The complete learning mechanism is:

$$
\boxed{
\text{Context Statistics}
\rightarrow
\text{Prediction Objective}
\rightarrow
\text{Softmax Probability}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradient Descent}
\rightarrow
\text{Embedding Matrix}
}
$$

The deepest idea is:

$$
\boxed{
\text{Word2Vec learns word representations by exploiting word--context relationships}
}
$$

and specifically:

$$
\boxed{
\text{Skip-gram: Center}\rightarrow\text{Context}
}
$$

$$
\boxed{
\text{CBOW: Context}\rightarrow\text{Center}
}
$$

# Negative Sampling

Negative Sampling is a technique used in Word2Vec to make the training objective computationally efficient when the vocabulary is large.

The key problem comes from the original Skip-gram formulation. Given a center word $w_c$, the model tries to predict a context word $w_o$ by learning:

$$
P(w_o\mid w_c)
$$

Using a full softmax over the vocabulary, this probability is defined as:

$$
\boxed{
P(w_o\mid w_c)
=
\frac{
\exp(v_{w_o}^Tv_{w_c})
}{
\sum_{w=1}^{V}\exp(v_w^Tv_{w_c})
}
}
$$

where:

- $v_{w_c}\in\mathbb R^d$ is the input embedding of the center word.
- $v_w\in\mathbb R^d$ is the output embedding of a candidate word.
- $V$ is the vocabulary size.
- $d$ is the embedding dimension.

The numerator represents the score of the observed context word:

$$
\exp(v_{w_o}^Tv_{w_c})
$$

while the denominator sums the scores of **all $V$ words in the vocabulary**:

$$
\sum_{w=1}^{V}\exp(v_w^Tv_{w_c})
$$

Therefore, the probability of the correct context word is its score normalized against every possible vocabulary word.

The main computational problem is precisely this denominator.

For every training example, the model must evaluate:

$$
v_1^Tv_{w_c},
v_2^Tv_{w_c},
\ldots,
v_V^Tv_{w_c}
$$

Thus, the computational cost is approximately:

$$
O(Vd)
$$

per prediction.

When $V$ is very large and the corpus contains millions of training examples, this becomes expensive and makes training slow.

Negative Sampling addresses this problem by changing the training objective rather than computing the full probability distribution over the vocabulary.

---

# 1. Core Idea

Instead of asking:

> Which word among the entire vocabulary is the correct context for $w_c$?

Negative Sampling asks:

> Given a center-context pair, is this pair a real pair or a randomly generated pair?

Therefore, the problem changes from a $V$-class classification problem into a **binary classification problem**.

Suppose:

$$
(w_c,w_o)
$$

is a genuine center-context pair extracted from the corpus.

This is called a **positive sample**:

$$
y=1
$$

We then randomly sample $K$ other words:

$$
w_1,w_2,\ldots,w_K
$$

and construct:

$$
(w_c,w_1),\ldots,(w_c,w_K)
$$

These are treated as **negative samples**:

$$
y=0
$$

For example, consider:

> The cat drinks milk.

Take `drinks` as the center word.

A positive pair is:

$$
(\text{drinks},\text{milk})
$$

Suppose the sampled negative words are `dog`, `car`, and `banana`.

Then the training examples become:

$$
(\text{drinks},\text{milk},1)
$$

$$
(\text{drinks},\text{dog},0)
$$

$$
(\text{drinks},\text{car},0)
$$

$$
(\text{drinks},\text{banana},0)
$$

The model therefore only needs to distinguish one real pair from a small number of fake pairs.

The central idea is:

$$
\boxed{
\text{Positive pair}
\rightarrow
\text{high score}
}
$$

$$
\boxed{
\text{Negative pair}
\rightarrow
\text{low score}
}
$$

---

# 2. Mathematical Formulation

For a center word $w_c$ and a candidate word $w$, define the compatibility score:

$$
s(w,w_c)=v_w^Tv_{w_c}
$$

This is simply the dot product between their embeddings.

A larger score means that the model considers the two words more compatible.

Because the task is now binary classification, we convert the score into a probability using the sigmoid function:

$$
\boxed{
\sigma(s)=\frac{1}{1+e^{-s}}
}
$$

Therefore:

$$
\boxed{
P(y=1\mid w_c,w)
=
\sigma(v_w^Tv_{w_c})
}
$$

For a positive pair $(w_c,w_o)$, we want:

$$
P(y=1\mid w_c,w_o)\rightarrow1
$$

which means:

$$
v_{w_o}^Tv_{w_c}\rightarrow+\infty
$$

For a negative pair $(w_c,w_k)$, we want:

$$
P(y=1\mid w_c,w_k)\rightarrow0
$$

which means:

$$
v_{w_k}^Tv_{w_c}\rightarrow-\infty
$$

Therefore, Negative Sampling explicitly encourages:

$$
\boxed{
\text{Positive pairs: increase compatibility}
}
$$

and:

$$
\boxed{
\text{Negative pairs: decrease compatibility}
}
$$

---

# 3. Negative Sampling Objective and Loss

Suppose we have one positive pair:

$$
(w_c,w_o)
$$

and $K$ negative samples:

$$
w_1,w_2,\ldots,w_K
$$

The objective for this training example is to maximize:

$$
\boxed{
\log\sigma(v_{w_o}^Tv_{w_c})
+
\sum_{k=1}^{K}
\log\sigma(-v_{w_k}^Tv_{w_c})
}
$$

The first term corresponds to the positive pair.

The second term corresponds to the negative pairs.

Equivalently, minimizing the negative log-likelihood gives the loss:

$$
\boxed{
L
=
-\log\sigma(v_{w_o}^Tv_{w_c})
-
\sum_{k=1}^{K}
\log\sigma(-v_{w_k}^Tv_{w_c})
}
$$

The two terms have clear meanings.

For the positive pair:

$$
-\log\sigma(v_{w_o}^Tv_{w_c})
$$

the model is encouraged to increase:

$$
v_{w_o}^Tv_{w_c}
$$

For each negative pair:

$$
-\log\sigma(-v_{w_k}^Tv_{w_c})
$$

the model is encouraged to decrease:

$$
v_{w_k}^Tv_{w_c}
$$

Hence:

$$
\boxed{
\text{Positive}
\rightarrow
\text{pull embeddings toward compatibility}
}
$$

$$
\boxed{
\text{Negative}
\rightarrow
\text{push embeddings toward incompatibility}
}
$$

This is the fundamental optimization mechanism behind Negative Sampling.

---

# 4. How the Gradient Learns the Embeddings

The embedding vectors are not manually arranged according to semantic meaning. They are learned through gradient descent.

For the positive pair, define:

$$
s_o=v_{w_o}^Tv_{w_c}
$$

The positive loss is:

$$
L_{pos}=-\log\sigma(s_o)
$$

Its derivative with respect to the score is:

$$
\boxed{
\frac{\partial L_{pos}}{\partial s_o}
=
\sigma(s_o)-1
}
$$

Suppose the model is currently making a poor prediction:

$$
\sigma(s_o)\approx0
$$

Then:

$$
\frac{\partial L_{pos}}{\partial s_o}\approx-1
$$

The gradient is large in magnitude, so the model strongly increases the compatibility of the positive pair.

If:

$$
\sigma(s_o)\approx1
$$

then:

$$
\frac{\partial L_{pos}}{\partial s_o}\approx0
$$

The pair is already classified correctly, so the update becomes small.

For a negative pair, define:

$$
s_k=v_{w_k}^Tv_{w_c}
$$

The negative loss is:

$$
L_{neg}=-\log\sigma(-s_k)
$$

and:

$$
\boxed{
\frac{\partial L_{neg}}{\partial s_k}
=
\sigma(s_k)
}
$$

If the negative word incorrectly has a high score:

$$
\sigma(s_k)\approx1
$$

then the gradient is large and the model strongly decreases the score.

If:

$$
\sigma(s_k)\approx0
$$

the negative pair is already well separated and the gradient is small.

Therefore:

$$
\boxed{
\text{Difficult pair}
\rightarrow
\text{large gradient}
\rightarrow
\text{large update}
}
$$

while:

$$
\boxed{
\text{Well-classified pair}
\rightarrow
\text{small gradient}
\rightarrow
\text{small update}
}
$$

The embeddings are then updated through gradient descent:

$$
\boxed{
E
\leftarrow
E-\alpha\frac{\partial L}{\partial E}
}
$$

After a very large number of such updates, words that repeatedly occur in similar contexts receive similar learning signals:

$$
\boxed{
\text{Similar Context Patterns}
\rightarrow
\text{Similar Gradient Updates}
\rightarrow
\text{Similar Embeddings}
}
$$

This is how the geometry of the word embedding space emerges.

---

# 5. Why Negative Sampling Is Computationally Efficient

The main advantage of Negative Sampling is that it avoids computing the full softmax normalization.

With full softmax, we must evaluate:

$$
\sum_{w=1}^{V}\exp(v_w^Tv_{w_c})
$$

which requires considering all $V$ vocabulary words.

The computational cost is approximately:

$$
O(Vd)
$$

With Negative Sampling, we only evaluate:

$$
1+K
$$

words:

- one positive word;
- $K$ sampled negative words.

Therefore the cost becomes approximately:

$$
O(Kd)
$$

where:

$$
K\ll V
$$

For example, if:

$$
V=100000
$$

and:

$$
K=5
$$

the full softmax considers:

$$
100000
$$

candidate words, whereas Negative Sampling considers only:

$$
1+5=6
$$

words for that training example.

Thus:

$$
\boxed{
\text{Full Softmax}
\rightarrow
O(Vd)
}
$$

while:

$$
\boxed{
\text{Negative Sampling}
\rightarrow
O(Kd)
}
$$

This is the fundamental reason Negative Sampling can make Word2Vec training much more practical for large vocabularies.

---

# 6. How Negative Samples Are Chosen

The negative words are not simply chosen uniformly at random.

Word2Vec uses a noise distribution related to word frequency. A commonly used form is:

$$
\boxed{
P_{noise}(w)\propto f(w)^{3/4}
}
$$

where $f(w)$ reflects the frequency of word $w$.

After normalization:

$$
P_{noise}(w)
=
\frac{
f(w)^{3/4}
}{
\sum_{w'}f(w')^{3/4}
}
$$

The exponent $3/4$ reduces the dominance of extremely frequent words while still preserving information from the corpus frequency distribution.

The resulting negative samples provide examples that the model should learn to distinguish from genuine word-context relationships.

---

# 7. Final Mental Model

The original Skip-gram objective is:

$$
\boxed{
P(w_o\mid w_c)
=
\frac{
\exp(v_{w_o}^Tv_{w_c})
}{
\sum_{w=1}^{V}\exp(v_w^Tv_{w_c})
}
}
$$

The problem is the denominator:

$$
\boxed{
\sum_{w=1}^{V}
}
$$

because it requires evaluating all vocabulary words for every training example.

Negative Sampling replaces this full-vocabulary computation with a much smaller binary classification problem:

$$
\boxed{
\text{One Positive Pair}
+
\text{$K$ Negative Pairs}
}
$$

Then:

$$
\boxed{
\text{Dot Product}
\rightarrow
\text{Sigmoid}
\rightarrow
\text{Binary Loss}
\rightarrow
\text{Gradient Descent}
\rightarrow
\text{Embedding Update}
}
$$

The complete conceptual picture is:

$$
\boxed{
\text{Real Word-Context Relationships}
\rightarrow
\text{Positive Samples}
}
$$

$$
\boxed{
\text{Randomly Generated Relationships}
\rightarrow
\text{Negative Samples}
}
$$

$$
\boxed{
\text{Positive}
\rightarrow
\text{Increase Compatibility}
\qquad
\text{Negative}
\rightarrow
\text{Decrease Compatibility}
}
$$

and after many training iterations:

$$
\boxed{
\text{Context Statistics}
\rightarrow
\text{Gradient Updates}
\rightarrow
\text{Embedding Geometry}
}
$$

The essential distinction to remember is:

$$
\boxed{
\text{Skip-gram}
=
\text{defines the center-to-context prediction task}
}
$$

whereas:

$$
\boxed{
\text{Negative Sampling}
=
\text{provides an efficient way to train that objective without evaluating the entire vocabulary}
}
$$

# GloVe Word Vectors

GloVe (**Global Vectors for Word Representation**) is a method for learning **distributed word representations** from a large text corpus.

The central idea is to exploit **word co-occurrence statistics**. Instead of learning word vectors primarily through a prediction task like Word2Vec, GloVe constructs a global word-context co-occurrence matrix and learns vectors that capture the structure of this matrix.

The core idea is:

$$
\boxed{
\text{Corpus}
\rightarrow
\text{Word-Context Co-occurrences}
\rightarrow
\text{Co-occurrence Matrix}
\rightarrow
\text{Optimization}
\rightarrow
\text{Word Vectors}
}
$$

The key distinction is:

$$
\boxed{
\text{Word2Vec: Predictive}
\qquad
\text{GloVe: Co-occurrence-based}
}
$$

---

# 1. Core Idea: Word Meaning from Co-occurrence

GloVe is based on the distributional hypothesis:

> A word's meaning is related to the words that tend to appear around it.

For example, consider:

> The cat drinks milk.

The words `cat` and `milk` may frequently occur in similar contexts involving:

> drink, food, animal, bowl, ...

Instead of explicitly assigning semantic labels such as:

$$
\text{cat}\rightarrow\text{animal}
$$

GloVe uses the statistical relationships between words in the corpus.

Suppose the vocabulary is:

$$
V=\{w_1,w_2,\ldots,w_{|V|}\}
$$

We construct a **word-context co-occurrence matrix**:

$$
X
$$

where:

$$
X_{ij}
$$

represents how frequently context word $w_j$ occurs within the context of word $w_i$.

For example:

| | cat | dog | milk | car |
|---|---:|---:|---:|---:|
| cat | 0 | 20 | 50 | 1 |
| dog | 20 | 0 | 35 | 0 |
| milk | 50 | 35 | 0 | 2 |
| car | 1 | 0 | 2 | 0 |

Thus:

$$
X_{\text{cat,milk}}=50
$$

means that `milk` occurs in the defined context of `cat` 50 times.

The important point is that GloVe does not treat these counts as the final embeddings. Instead, these statistics provide the **learning signal** from which the embeddings are learned.

---

# 2. From Co-occurrence Counts to Probabilities

For a given target word $w_i$, define:

$$
X_i=\sum_k X_{ik}
$$

where $X_i$ is the total number of context-word occurrences associated with $w_i$.

The conditional probability of observing context word $w_j$ given target word $w_i$ is:

$$
\boxed{
P_{ij}
=
P(w_j\mid w_i)
=
\frac{X_{ij}}{X_i}
}
$$

This converts raw counts into probabilities.

For example, suppose:

$$
X_{\text{ice}}=1000
$$

and:

$$
X_{\text{ice,cold}}=300
$$

Then:

$$
P(\text{cold}\mid\text{ice})
=
\frac{300}{1000}
=
0.3
$$

The absolute probability can already provide useful information, but GloVe emphasizes that **relative probabilities between context words are especially informative**.

For a target word $w_i$ and two context words $w_j$ and $w_k$, consider:

$$
\boxed{
\frac{P_{ik}}{P_{ij}}
}
$$

This ratio tells us how strongly $w_i$ is associated with $w_k$ relative to $w_j$.

For example, consider the target words:

$$
\text{ice}
\qquad\text{and}\qquad
\text{steam}
$$

and context words:

$$
\text{solid}
\qquad\text{gas}
$$

We can compare:

$$
\frac{P(\text{solid}\mid\text{ice})}
{P(\text{gas}\mid\text{ice})}
$$

with:

$$
\frac{P(\text{solid}\mid\text{steam})}
{P(\text{gas}\mid\text{steam})}
$$

The differences between such ratios contain information about the relationships between words.

This motivates GloVe's objective.

---

# 3. GloVe Model and Objective Function

GloVe assigns two vectors to each word:

$$
w_i\in\mathbb R^d
$$

and:

$$
\tilde w_j\in\mathbb R^d
$$

where:

- $w_i$ represents the target-word vector;
- $\tilde w_j$ represents the context-word vector.

It also introduces two bias terms:

$$
b_i
\qquad\text{and}\qquad
\tilde b_j
$$

The model assumes that the relationship between a target word $w_i$ and context word $w_j$ can be represented by:

$$
w_i^T\tilde w_j+b_i+\tilde b_j
$$

GloVe trains these parameters so that this quantity approximates the logarithm of the co-occurrence count:

$$
\boxed{
w_i^T\tilde w_j+b_i+\tilde b_j
\approx
\log X_{ij}
}
$$

Therefore, the objective function is:

$$
\boxed{
J
=
\sum_{i,j}
f(X_{ij})
\left(
w_i^T\tilde w_j+b_i+\tilde b_j-\log X_{ij}
\right)^2
}
$$

This is a **weighted least-squares objective**.

For one pair $(i,j)$, define the prediction error:

$$
E_{ij}
=
w_i^T\tilde w_j+b_i+\tilde b_j-\log X_{ij}
$$

Then:

$$
L_{ij}
=
f(X_{ij})E_{ij}^2
$$

The optimizer tries to minimize this error:

$$
E_{ij}\rightarrow0
$$

which means:

$$
w_i^T\tilde w_j+b_i+\tilde b_j
\rightarrow
\log X_{ij}
$$

Thus, the word vectors are learned so that their geometric relationships reflect the statistical structure of word co-occurrences.

---

# 4. Why Does GloVe Use $\log X_{ij}$?

The use of the logarithm is important.

Raw co-occurrence counts can vary dramatically:

$$
X_{ij}=1
$$

versus:

$$
X_{ik}=100000
$$

If the model tried to directly learn:

$$
w_i^T\tilde w_j\approx X_{ij}
$$

very large counts could dominate the objective.

Instead, GloVe learns:

$$
w_i^T\tilde w_j
\approx
\log X_{ij}
$$

The logarithm compresses the scale:

$$
\log(1)=0
$$

while:

$$
\log(100000)\approx11.51
$$

rather than preserving the enormous difference between $1$ and $100000$ directly.

Another useful property is:

$$
\log(ab)=\log a+\log b
$$

which makes logarithmic co-occurrence statistics naturally compatible with additive and dot-product representations.

The key idea is:

$$
\boxed{
\text{Raw Count}
\rightarrow
\text{Log Count}
\rightarrow
\text{More Suitable Learning Target}
}
$$

---

# 5. The Weighting Function $f(X_{ij})$

The objective contains:

$$
f(X_{ij})
$$

because not every co-occurrence observation should contribute equally.

Extremely rare co-occurrences may be unreliable, while extremely frequent ones should not dominate the entire optimization.

A common form is:

$$
\boxed{
f(x)=
\begin{cases}
\left(\frac{x}{x_{\max}}\right)^\alpha,
& x<x_{\max}\\
1,
& x\ge x_{\max}
\end{cases}
}
$$

where $x_{\max}$ and $\alpha$ are hyperparameters.

The weighting function therefore controls the contribution of different co-occurrence counts.

The overall objective becomes:

$$
\boxed{
J
=
\sum_{i,j}
f(X_{ij})
\left(
w_i^T\tilde w_j+b_i+\tilde b_j-\log X_{ij}
\right)^2
}
$$

So GloVe does not simply memorize the co-occurrence matrix. It finds a lower-dimensional representation that captures its important statistical structure.

---

# 6. How the Word Vectors Are Learned

The training process can be understood as repeatedly reducing the discrepancy between:

$$
w_i^T\tilde w_j+b_i+\tilde b_j
$$

and:

$$
\log X_{ij}
$$

For each observed word-context pair:

$$
(i,j)
$$

the model computes the loss:

$$
L_{ij}
=
f(X_{ij})
\left(
w_i^T\tilde w_j+b_i+\tilde b_j-\log X_{ij}
\right)^2
$$

Then backpropagation computes gradients:

$$
\frac{\partial L}{\partial w_i},
\qquad
\frac{\partial L}{\partial \tilde w_j},
\qquad
\frac{\partial L}{\partial b_i},
\qquad
\frac{\partial L}{\partial\tilde b_j}
$$

and gradient descent updates the parameters:

$$
\boxed{
\theta
\leftarrow
\theta-\alpha\nabla_\theta L
}
$$

After repeatedly processing the co-occurrence statistics:

$$
\boxed{
\text{Co-occurrence Matrix}
\rightarrow
\text{Optimization}
\rightarrow
\text{Learned Word Vectors}
}
$$

The important point is that GloVe is learning a representation of the **relationships encoded in the co-occurrence matrix**, not simply storing the matrix itself.

---

# 7. Why Semantic Structure Emerges

Suppose two words have similar co-occurrence patterns:

$$
X_{i,:}\approx X_{k,:}
$$

That means they tend to occur with similar context words.

Because GloVe tries to encode these statistics in vector form, similar co-occurrence structures tend to result in related vectors.

Thus:

$$
\boxed{
\text{Similar Context Patterns}
\rightarrow
\text{Similar Statistical Constraints}
\rightarrow
\text{Similar Embedding Geometry}
}
$$

This allows the embedding space to capture semantic and syntactic relationships.

For example, the vector differences between related words can sometimes encode relationships such as:

$$
w_{\text{king}}-w_{\text{queen}}
$$

and:

$$
w_{\text{man}}-w_{\text{woman}}
$$

being approximately related.

These relationships emerge from the statistical structure of the corpus rather than being explicitly programmed into the model.

---

# 8. GloVe vs. Word2Vec

The most important difference is how the training signal is constructed.

Word2Vec creates a prediction problem from word-context pairs.

For Skip-gram:

$$
\boxed{
\text{Center Word}
\rightarrow
\text{Context Word}
}
$$

The model learns embeddings because they are useful for predicting surrounding words.

GloVe instead starts by constructing global co-occurrence statistics:

$$
\boxed{
X_{ij}
=
\text{co-occurrence count of }w_i\text{ and }w_j
}
$$

and learns vectors that model:

$$
\boxed{
\log X_{ij}
}
$$

through the weighted least-squares objective.

Therefore:

| Property | Word2Vec | GloVe |
|---|---|---|
| Main approach | Predictive | Co-occurrence-based |
| Learning signal | Context prediction | Co-occurrence statistics |
| Core quantity | $P(w_o\mid w_c)$ | $X_{ij}$ and $\log X_{ij}$ |
| Training objective | Prediction objective | Weighted least squares |
| Main view | Learn by predicting | Learn from global statistics |

The key distinction is:

$$
\boxed{
\text{Word2Vec}
\rightarrow
\text{learn representations through prediction}
}
$$

while:

$$
\boxed{
\text{GloVe}
\rightarrow
\text{learn representations by modeling co-occurrence statistics}
}
$$

---

# 9. Final Mental Model

GloVe begins with raw text:

$$
\boxed{
\text{Large Corpus}
}
$$

Then constructs word-context statistics:

$$
\boxed{
\text{Corpus}
\rightarrow
\text{Co-occurrence Matrix }X
}
$$

From the matrix, the model obtains:

$$
\boxed{
P(w_j\mid w_i)
=
\frac{X_{ij}}{X_i}
}
$$

and uses the statistical structure of these co-occurrences to learn word vectors.

The core GloVe relationship is:

$$
\boxed{
w_i^T\tilde w_j+b_i+\tilde b_j
\approx
\log X_{ij}
}
$$

with objective:

$$
\boxed{
J
=
\sum_{i,j}
f(X_{ij})
\left(
w_i^T\tilde w_j+b_i+\tilde b_j-\log X_{ij}
\right)^2
}
$$

After optimization:

$$
\boxed{
\text{Co-occurrence Statistics}
\rightarrow
\text{Learned Vectors}
\rightarrow
\text{Embedding Geometry}
}
$$

The deepest idea to remember is:

$$
\boxed{
\text{GloVe learns word vectors by encoding global word-context co-occurrence patterns}
}
$$

while the central comparison is:

$$
\boxed{
\text{Word2Vec}
=
\text{Predictive}
}
$$

$$
\boxed{
\text{GloVe}
=
\text{Co-occurrence-based}
}
$$